In [1]:
import awkward as ak
from coffea import nanoevents
from coffea.analysis_tools import PackedSelection
from coffea.nanoevents.methods import vector

ak.behavior.update(vector.behavior)

import os
import pickle

import matplotlib.pyplot as plt
import numpy as np


/uscms_data/d3/ammitra/XHYbbWW_analysis/CMSSW_12_3_5/src/timber-env/lib/python3.9/site-packages/coffea/nanoevents/schemas/fcc.py:5: FutureWarning: In version 2025.1.0 (target date: 2024-12-31 11:59:59-06:00), this will be an error.
To raise these warnings as errors (and get stack traces to find out where they're called), run
    import warnings
    warnings.filterwarnings("error", module="coffea.*")
after the first `import coffea` or use `@pytest.mark.filterwarnings("error:::coffea.*")` in pytest.
Issue: coffea.nanoevents.methods.vector will be removed and replaced with scikit-hep vector. Nanoevents schemas internal to coffea will be migrated. Otherwise please consider using that package!.
  from coffea.nanoevents.methods import vector
/uscms_data/d3/ammitra/XHYbbWW_analysis/CMSSW_12_3_5/src/timber-env/lib/python3.9/site-packages/numpy/core/getlimits.py:499: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, 

In [2]:
d_PDGID = 1
u_PDGID = 2
s_PDGID = 3
c_PDGID = 4
b_PDGID = 5
g_PDGID = 21
TOP_PDGID = 6

ELE_PDGID = 11
vELE_PDGID = 12
MU_PDGID = 13
vMU_PDGID = 14
TAU_PDGID = 15
vTAU_PDGID = 16

G_PDGID = 22
Z_PDGID = 23
W_PDGID = 24
HIGGS_PDGID = 25
Y_PDGID = 35

b_PDGIDS = [511, 521, 523]

GRAV_PDGID = 39

GEN_FLAGS = ["fromHardProcess", "isLastCopy"]

In [4]:
plot_dir = 'plots/'

In [5]:
P4 = {
    "eta": "Eta",
    "phi": "Phi",
    "mass": "Mass",
    "pt": "Pt",
}


PAD_VAL = -99999


skim_vars = {
    "FatJet": {
        **P4,
        "msoftdrop": "Msd",
        "particleNetMD_QCD": "ParticleNetMD_QCD",
        "particleNetMD_Xbb": "ParticleNetMD_Xbb",
        "particleNet_H4qvsQCD": "ParticleNet_Th4q",
        "particleNet_mass": "ParticleNetMass",
    },
    "GenHiggs": P4,
    "other": {"MET_pt": "MET_pt"},
}


def pad_val(
    arr: ak.Array,
    target: int,
    value: float = PAD_VAL,
    axis: int = 0,
    to_numpy: bool = True,
    clip: bool = True,
):
    """
    pads awkward array up to ``target`` index along axis ``axis`` with value ``value``,
    optionally converts to numpy array
    """
    ret = ak.fill_none(ak.pad_none(arr, target, axis=axis, clip=clip), value, axis=axis)
    return ret.to_numpy() if to_numpy else ret


def add_selection(
    name: str,
    sel: np.ndarray,
    selection: PackedSelection,
    cutflow: dict,
    isData: bool,
    genWeights: ak.Array = None,
):
    """adds selection to PackedSelection object and the cutflow dictionary"""
    if isinstance(sel, ak.Array):
        sel = sel.to_numpy()

    selection.add(name, sel.astype(bool))
    cutflow[name] = (
        np.sum(selection.all(*selection.names))
        if isData
        # add up genWeights for MC
        else np.sum(genWeights[selection.all(*selection.names)])
    )


In [63]:
events = nanoevents.NanoEventsFactory.from_root(
    #file='root://cmseos.fnal.gov//store/user/ammitra/XHYbbWW/snapshots/HWWsnapshot_NMSSM-XHY-1800-1200_18_*of2.root',
    #'root://cmseos.fnal.gov//store/user/ammitra/XHYbbWW/snapshots/HWWsnapshot_NMSSM-XHY-1800-1200_18_2of2.root:Events',
    #treepath='Events',
    {"root://cmseos.fnal.gov//store/user/ammitra/XHYbbWW/snapshots/HWWsnapshot_NMSSM-XHY-1800-1200_18_*of2.root":"Events"},
    schemaclass=nanoevents.NanoAODSchema,
).events()



In [67]:
genparts = events.GenPart[events.GenPart.hasFlags(GEN_FLAGS)]


In [68]:
genqs = events.GenPart[
    (np.abs(events.GenPart.pdgId) <= b_PDGID) * events.GenPart.hasFlags(GEN_FLAGS)
]


In [69]:
genvs = events.GenPart[
    ((np.abs(events.GenPart.pdgId) == W_PDGID) | (np.abs(events.GenPart.pdgId) == Z_PDGID))
    * events.GenPart.hasFlags(GEN_FLAGS)
]

In [72]:
skim_vars.items()

dict_items([('eta', 'Eta'), ('phi', 'Phi'), ('mass', 'Mass'), ('pt', 'Pt')])

In [90]:
{f"GenHiggs{key}": higgs[var].to_numpy() for (var, key) in skim_vars.items()}

TypeError: Converting from an nplike without known data to an nplike with known data is not supported

This error occurred while calling

    ak.to_numpy(
        <Array-typetracer [...] type='## * var * float32[parameters={"__doc...'>
        allow_missing = True
    )

In [70]:
skim_vars = P4
fatjets = events.Trijet

higgs = events.GenPart[
    (abs(events.GenPart.pdgId) == HIGGS_PDGID) * events.GenPart.hasFlags(GEN_FLAGS)
]
GenHiggsVars = {f"GenHiggs{key}": higgs[var].to_numpy() for (var, key) in skim_vars.items()}
is_bb = abs(higgs.children.pdgId) == b_PDGID
has_bb = ak.sum(ak.flatten(is_bb, axis=2), axis=1) == 2

bb = ak.flatten(higgs.children[is_bb], axis=2)
GenbbVars = {f"Genbb{key}": pad_val(bb[var], 2, axis=1) for (var, key) in skim_vars.items()}

# gen Y and kids
Ys = events.GenPart[(abs(events.GenPart.pdgId) == Y_PDGID) * events.GenPart.hasFlags(GEN_FLAGS)]
GenYVars = {f"GenY{key}": Ys[var].to_numpy() for (var, key) in skim_vars.items()}
is_VV = (abs(Ys.children.pdgId) == W_PDGID) + (abs(Ys.children.pdgId) == Z_PDGID)
has_VV = ak.sum(ak.flatten(is_VV, axis=2), axis=1) == 2

add_selection("has_bbVV", has_bb * has_VV, selection, cutflow, False, signGenWeights)

VV = ak.flatten(Ys.children[is_VV], axis=2)
GenVVVars = {f"GenVV{key}": VV[var][:, :2].to_numpy() for (var, key) in skim_vars.items()}

VV_children = VV.children

# iterate through the children in photon scattering events to get final daughter quarks
for i in range(5):
    photon_mask = ak.any(ak.flatten(abs(VV_children.pdgId), axis=2) == G_PDGID, axis=1)
    if not np.any(photon_mask):
        break

    # use a where condition to get next layer of children for photon scattering events
    VV_children = ak.where(photon_mask, ak.flatten(VV_children.children, axis=3), VV_children)

quarks = abs(VV_children.pdgId) <= b_PDGID
all_q = ak.all(ak.all(quarks, axis=2), axis=1)
add_selection("all_q", all_q, selection, cutflow, False, signGenWeights)

V_has_2q = ak.count(VV_children.pdgId, axis=2) == 2
has_4q = ak.values_astype(ak.prod(V_has_2q, axis=1), bool)
add_selection("has_4q", has_4q, selection, cutflow, False, signGenWeights)

Gen4qVars = {
    f"Gen4q{key}": ak.to_numpy(
        ak.fill_none(
            ak.pad_none(ak.pad_none(VV_children[var], 2, axis=1, clip=True), 2, axis=2, clip=True),
            PAD_VAL,
        )
    )
    for (var, key) in skim_vars.items()
}

# fatjet gen matching
Hbb = ak.pad_none(higgs, 1, axis=1, clip=True)[:, 0]
HVV = ak.pad_none(Ys, 1, axis=1, clip=True)[:, 0]

bbdr = fatjets[:, :2].delta_r(Hbb)
vvdr = fatjets[:, :2].delta_r(HVV)

match_dR = 0.8
Hbb_match = bbdr <= match_dR
HVV_match = vvdr <= match_dR

# overlap removal - in the case where fatjet is matched to both, match it only to the closest Higgs
Hbb_match = (Hbb_match * ~HVV_match) + (bbdr <= vvdr) * (Hbb_match * HVV_match)
HVV_match = (HVV_match * ~Hbb_match) + (bbdr > vvdr) * (Hbb_match * HVV_match)

VVJets = ak.pad_none(fatjets[HVV_match], 1, axis=1)[:, 0]
quarkdrs = ak.flatten(VVJets.delta_r(VV_children), axis=2)
num_prongs = ak.sum(quarkdrs < match_dR, axis=1)

GenMatchingVars = {
    "ak8FatJetHbb": pad_val(Hbb_match, 2, axis=1),
    "ak8FatJetHVV": pad_val(HVV_match, 2, axis=1),
    "ak8FatJetHVVNumProngs": ak.fill_none(num_prongs, PAD_VAL).to_numpy(),
}

genbb, gen4q = bb, ak.flatten(VV_children, axis=2)

TypeError: Converting from an nplike without known data to an nplike with known data is not supported

This error occurred while calling

    ak.to_numpy(
        <Array-typetracer [...] type='## * var * float32[parameters={"__doc...'>
        allow_missing = True
    )